In [3]:
import json
import networkx as nx

with open("crime_kg_nodes_edges.json", "r", encoding="utf-8") as f:
    kg_data = json.load(f)

G = nx.Graph()
for node in kg_data["nodes"]:
    node_id = node["id"]
    attrs = {k: v for k, v in node.items() if k != "id"}
    G.add_node(node_id, **attrs)
for edge in kg_data["edges"]:
    G.add_edge(edge["source"], edge["target"], relationship=edge["relationship"])

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

Nodes: 1530 Edges: 2127


In [9]:
import numpy as np
import torch

nodes = list(G.nodes())
node_index = {node_id: i for i, node_id in enumerate(nodes)}

NODE_TYPES = ["CASE", "SUSPECT", "CRIME_TYPE", "POLICE_BEAT", "LOCATION", "VEHICLE"]

max_degree = max(dict(G.degree()).values())

features = []
labels = []
suspect_mask_all = []

for n in nodes:
    node_type = G.nodes[n].get("type")
    type_onehot = [1.0 if node_type == t else 0.0 for t in NODE_TYPES]
    degree_norm = G.degree(n) / max_degree

    features.append(type_onehot + [degree_norm])

    if node_type == "SUSPECT":
        case_count = sum(1 for nb in G.neighbors(n) if G.nodes[nb].get("type") == "CASE")
        labels.append(1 if case_count >= 2 else 0)
        suspect_mask_all.append(True)
    else:
        labels.append(0)
        suspect_mask_all.append(False)

x = torch.tensor(features, dtype=torch.float)
y = torch.tensor(labels, dtype=torch.long)
suspect_mask_all = torch.tensor(suspect_mask_all, dtype=torch.bool)

edges_idx = [(node_index[u], node_index[v]) for u, v in G.edges()]
edge_index = torch.tensor(
    edges_idx + [(b, a) for a, b in edges_idx], dtype=torch.long
).t().contiguous()

print("x shape:", x.shape)
print("edge_index shape:", edge_index.shape)
print("Total suspects:", suspect_mask_all.sum().item())
print("Suspicious (label=1) among suspects:", y[suspect_mask_all].sum().item())

x shape: torch.Size([1530, 7])
edge_index shape: torch.Size([2, 4254])
Total suspects: 434
Suspicious (label=1) among suspects: 61


In [1]:
%pip install torch_geometric

  Using cached torch_geometric-2.8.0.post1-py3-none-any.whl.metadata (64 kB)
  Using cached aiohttp-3.14.3-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-4.0.1-cp314-cp314-win_amd64.whl.metadata (18 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.5.2-cp314-cp314-win_amd64.whl.metadata (17 kB)
  Using cached yarl-1.24.5-cp314-cp314-win_amd64.whl.metadata (107 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 5.9 MB/s  0:00:00
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)

   ---- -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
import torch
from sklearn.model_selection import train_test_split

suspect_indices = torch.where(suspect_mask_all)[0].tolist()
suspect_labels = y[suspect_indices].tolist()

train_idx, val_idx = train_test_split(
    suspect_indices,
    test_size=0.2,
    random_state=42,
    stratify=suspect_labels
)

train_mask = torch.zeros(len(nodes), dtype=torch.bool)
val_mask = torch.zeros(len(nodes), dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True

print("Train suspects:", train_mask.sum().item(), "| suspicious:", y[train_mask].sum().item())
print("Val suspects:", val_mask.sum().item(), "| suspicious:", y[val_mask].sum().item())

Train suspects: 347 | suspicious: 49
Val suspects: 87 | suspicious: 12


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask
data.val_mask = val_mask

class GraphSAGE(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.sage1 = SAGEConv(input_features, 32)
        self.sage2 = SAGEConv(32, 32)
        self.classifier = nn.Linear(32, 2)

    def forward(self, x, edge_index):
        x = self.sage1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.sage2(x, edge_index)
        x = F.relu(x)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)
model = GraphSAGE(data.num_node_features).to(device)

print("Model ready. Device:", device)
print("Input features:", data.num_node_features)

Model ready. Device: cpu
Input features: 7


In [18]:
class_weights = torch.tensor([1.0, 2.0], dtype=torch.float).to(device)
loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

from sklearn.metrics import f1_score

EPOCHS = 200
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    output = model(data.x, data.edge_index)
    loss = loss_function(output[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            val_output = model(data.x, data.edge_index)
            val_pred = val_output[data.val_mask].argmax(dim=1)
            val_actual = data.y[data.val_mask]
            f1 = f1_score(val_actual.cpu(), val_pred.cpu(), zero_division=0)
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Val F1: {f1:.4f}")
        

Epoch 20 | Loss: 0.1233 | Val F1: 1.0000
Epoch 40 | Loss: 0.0277 | Val F1: 1.0000
Epoch 60 | Loss: 0.0212 | Val F1: 1.0000
Epoch 80 | Loss: 0.0171 | Val F1: 1.0000
Epoch 100 | Loss: 0.0208 | Val F1: 1.0000
Epoch 120 | Loss: 0.0152 | Val F1: 1.0000
Epoch 140 | Loss: 0.0156 | Val F1: 1.0000
Epoch 160 | Loss: 0.0175 | Val F1: 1.0000
Epoch 180 | Loss: 0.0156 | Val F1: 0.3038
Epoch 200 | Loss: 0.0150 | Val F1: 1.0000


In [16]:
model.eval()
with torch.no_grad():
    val_output = model(data.x, data.edge_index)
    val_pred = val_output[data.val_mask].argmax(dim=1)
    val_actual = data.y[data.val_mask]

print("Predicted suspicious count:", val_pred.sum().item(), "/", len(val_pred))
print("Actual suspicious count   :", val_actual.sum().item(), "/", len(val_actual))

Predicted suspicious count: 25 / 87
Actual suspicious count   : 12 / 87


In [19]:
torch.save(model.state_dict(), "graphsage_model.pt")
print("GraphSAGE model saved.")

model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    probabilities = F.softmax(logits, dim=1)
    predictions = logits.argmax(dim=1)

print("\nSample suspect risk scores:")
count = 0
for i, node_id in enumerate(nodes):
    if suspect_mask_all[i] and count < 10:
        prob = probabilities[i][1].item()
        status = "SUSPICIOUS" if predictions[i].item() == 1 else "NORMAL"
        print(f"{node_id:25s} | {status:10s} | risk: {prob:.3f}")
        count += 1

GraphSAGE model saved.

Sample suspect risk scores:
Deandre Allen             | NORMAL     | risk: 0.002
Tyler Wilson              | SUSPICIOUS | risk: 1.000
Devon Wright              | NORMAL     | risk: 0.414
Jerome Washington         | NORMAL     | risk: 0.406
Brandon Allen             | SUSPICIOUS | risk: 0.998
Darius Williams           | SUSPICIOUS | risk: 0.998
Lamar Hernandez           | NORMAL     | risk: 0.394
Derek Vance               | NORMAL     | risk: 0.399
Marcus Thompson           | NORMAL     | risk: 0.444
Tyler Nguyen              | NORMAL     | risk: 0.390
